# model_wrapper

> `TiledModel` — the single user-facing `nn.Module`.
>
> Users provide a small-input PyTorch model, a model path (or a custom loader),
> and configuration objects.  `TiledModel` handles all tiling, batched inference,
> and Gaussian-weighted stitching internally.  From the outside it looks like a
> standard `nn.Module` that accepts `(1, C, H, W)` and returns `(1, C_out, H, W)`.

In [ ]:
#| default_exp model_wrapper

In [2]:
%load_ext autoreload    
%autoreload 2

In [5]:
#| hide
from nbdev.showdoc import *

In [11]:
#| export
from __future__ import annotations

import logging
import warnings
from pathlib import Path
from typing import Callable, List, Optional, Tuple, Union

import torch
import torch.nn as nn
from torch import Tensor


In [13]:
#| exporti
from segmentation_tools_pt.tiling import *
from segmentation_tools_pt.stitching import *


In [16]:
#| export
logger = logging.getLogger(__name__)

# Public type alias so callers can document custom loaders
ModelLoaderFn = Callable[[str], nn.Module]

## Default model loader

In [17]:
#| export
def default_model_loader(model_path: str) -> nn.Module:
    """Load a ``torch.save``'d ``nn.Module`` from disk.

    Uses ``weights_only=False`` to support full model checkpoints
    (not just state-dicts).  If you only saved a state-dict, subclass
    :class:`TiledModel` and override :meth:`TiledModel.from_path` to
    supply your own loader.

    Args:
        model_path: Absolute or relative path to the ``.pt`` / ``.pth`` file.

    Returns:
        Loaded ``nn.Module`` in evaluation mode.

    Raises:
        FileNotFoundError: If ``model_path`` does not exist.
    """
    path = Path(model_path)
    if not path.exists():
        raise FileNotFoundError(f"Model file not found: {path}")

    # weights_only=False required for full-model checkpoints.
    # Prefer loading from trusted sources only.
    model: nn.Module = torch.load(path, map_location="cpu", weights_only=False)
    model.eval()
    logger.debug("default_model_loader: loaded %s", path)
    return model

## TiledInferenceModelONNX

ONNX opset 12 compatible version.  Composes `TileProcessorONNX` →
model → `TileMergerONNX` with no Python dataclasses, lists, or
dynamic control flow in `forward()`.


In [ ]:
#| export
class TiledInferenceModelONNX(nn.Module):
    """ONNX opset 12 compatible tiling-inference-stitching pipeline.

    Composes :class:`~segmentation_tools_pt.tiling.TileProcessorONNX` (tiler),
    a user-provided segmentation model, and
    :class:`~segmentation_tools_pt.stitching.TileMergerONNX` (merger).

    All tile positions are pre-computed at construction time.  ``forward()``
    is pure tensor ops — no Python dataclasses, lists-of-coords, or dynamic
    control flow — so it traces cleanly to an ONNX graph.

    """

    def __init__(
        self,
        model:                nn.Module, # Underlying segmentation model
        tile_size:            int   = 256, # Tile side length
        overlap:              int   = 32, # Overlap between adjacent tiles
        input_h:              int   = 1152, # Fixed input image height
        input_w:              int   = 1632, # Fixed input image width
        merge_strategy:       str   = "gaussian_weighted", # Merge strategy
        device:               str   = 'cpu',
        confidence_threshold: float = 0.5, # Binarisation threshold in the merger
        apply_sigmoid:        bool  = False, # Whether to apply sigmoid to model outputs before merging
        apply_softmax:        bool  = False, # Whether to apply softmax to model outputs before merging
        postprocessing:       str | list | bool | None = "light", # Preset or custom pipeline
        apply_postprocessing: Optional[bool] = None, # Deprecated — use postprocessing instead
    ) -> None:
        super().__init__()

        if apply_sigmoid and apply_softmax:
            raise ValueError("apply_sigmoid and apply_softmax cannot both be True")

        if apply_postprocessing is not None:
            warnings.warn(
                "apply_postprocessing is deprecated; use postprocessing='light'|'none'|... instead.",
                DeprecationWarning,
                stacklevel=2,
            )
            postprocessing = "light" if apply_postprocessing else "none"

        pipeline = build_postprocessing_pipeline(postprocessing)

        self.model = model
        self.apply_sigmoid = apply_sigmoid
        self.apply_softmax = apply_softmax
        self.tiler = TileProcessorONNX(
            tile_size=tile_size, overlap=overlap,
            input_h=input_h, input_w=input_w,
        )
        self.merger = TileMergerONNX(
            tile_size=tile_size, overlap=overlap,
            input_h=input_h, input_w=input_w,
            merge_strategy=merge_strategy,
            device=device,
            confidence_threshold=confidence_threshold,
            postprocessing_pipeline=pipeline,
        )

    def forward(self, x: Tensor) -> Tensor:
        """Run tiled inference.

        Args:
            x: ``(1, C, input_h, input_w)``

        Returns:
            ``(1, C_out, input_h, input_w)``
        """
        # 1. Split into tiles — [N, C, tile_size, tile_size]
        tiles = self.tiler(x)

        # 2. Run model on all tiles at once
        preds = self.model(tiles)
        # 3. Convert logits -> probabilities if needed
        if self.apply_sigmoid:
            preds = torch.sigmoid(preds)
        elif self.apply_softmax:
            preds = torch.softmax(preds, dim=1)

        # 4. Merge predictions back to full resolution
        return self.merger(preds)

## DualOutputTiledInferenceModelONNX

For models that return **two outputs** — e.g. segmentation `(N, C_seg, H, W)` +
presence/classification `(N, C_cls)`.

- The segmentation head is merged spatially (same as `TiledInferenceModelONNX`).
- The presence head is **aggregated** across tiles via `max` (default) or `mean`.

`max` is correct for defect/wound detection: if *any* tile detects the object,
the image-level label should be positive.


In [ ]:
#| export
class DualOutputTiledInferenceModelONNX(nn.Module):
    """ONNX opset 12 tiling pipeline for models returning (segmentation, presence).

    The wrapped model's ``forward()`` must return exactly two tensors:
      - seg:      ``(N, C_seg, tile_size, tile_size)`` — spatial predictions
      - presence: ``(N, C_cls)`` — per-tile scalar classification/presence logits

    Segmentation tiles are merged spatially (Gaussian/confidence weighted).
    Presence logits are aggregated across tiles via ``max`` or ``mean``.

    Returns:
        Tuple of:
          - merged_seg:   ``(1, C_seg, input_h, input_w)``
          - agg_presence: ``(1, C_cls)``
    """

    VALID_PRESENCE_AGG = ("max", "mean")

    def __init__(
        self,
        model:                  nn.Module,  # Model returning (seg, presence)
        tile_size:              int   = 256,  # Tile side length
        overlap:                int   = 32,  # Overlap between adjacent tiles
        input_h:                int   = 1152,  # Fixed input image height
        input_w:                int   = 1632,  # Fixed input image width
        merge_strategy:         str   = "gaussian_weighted",  # Spatial merge strategy
        presence_aggregation:   str   = "max",  # "max" or "mean" for presence head
        device:                 str   = "cpu",
        confidence_threshold:   float = 0.5,  # Binarisation threshold in the merger
        apply_sigmoid:          bool  = False,  # Apply sigmoid to seg output before merging
        apply_softmax:          bool  = False,  # Apply softmax to seg output before merging
        postprocessing:         str | list | bool | None = "light",  # Preset or custom pipeline
        apply_postprocessing:   Optional[bool] = None,  # Deprecated — use postprocessing instead
    ) -> None:
        super().__init__()

        if apply_sigmoid and apply_softmax:
            raise ValueError("apply_sigmoid and apply_softmax cannot both be True")
        if presence_aggregation not in self.VALID_PRESENCE_AGG:
            raise ValueError(
                f"presence_aggregation must be one of {self.VALID_PRESENCE_AGG}, "
                f"got {presence_aggregation!r}"
            )

        if apply_postprocessing is not None:
            warnings.warn(
                "apply_postprocessing is deprecated; use postprocessing='light'|'none'|... instead.",
                DeprecationWarning,
                stacklevel=2,
            )
            postprocessing = "light" if apply_postprocessing else "none"

        pipeline = build_postprocessing_pipeline(postprocessing)

        self.model = model
        self.apply_sigmoid = apply_sigmoid
        self.apply_softmax = apply_softmax
        self.presence_aggregation = presence_aggregation

        self.tiler = TileProcessorONNX(
            tile_size=tile_size, overlap=overlap,
            input_h=input_h, input_w=input_w,
        )
        self.merger = TileMergerONNX(
            tile_size=tile_size, overlap=overlap,
            input_h=input_h, input_w=input_w,
            merge_strategy=merge_strategy,
            device=device,
            confidence_threshold=confidence_threshold,
            postprocessing_pipeline=pipeline,
        )

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor]:
        """Run tiled inference with dual-output model.

        Args:
            x: ``(1, C, input_h, input_w)``

        Returns:
            Tuple of:
              - merged_seg:   ``(1, C_seg, input_h, input_w)``
              - agg_presence: ``(1, C_cls)``
        """
        # 1. Split into tiles — (N, C, tile_size, tile_size)
        tiles = self.tiler(x)

        # 2. Run model — returns (seg_preds, presence)
        seg_preds, presence = self.model(tiles)

        # 3. Convert seg logits -> probabilities if needed
        if self.apply_sigmoid:
            seg_preds = torch.sigmoid(seg_preds)
        elif self.apply_softmax:
            seg_preds = torch.softmax(seg_preds, dim=1)

        # 4. Merge segmentation spatially
        merged_seg = self.merger(seg_preds)

        # 5. Aggregate presence across tiles
        if self.presence_aggregation == "max":
            agg_presence = presence.max(dim=0, keepdim=True).values  # (1, C_cls)
        else:  # mean
            agg_presence = presence.mean(dim=0, keepdim=True)  # (1, C_cls)

        return merged_seg, agg_presence

## Smoke tests

In [28]:
#| test
import torch
import torch.nn as nn

INPUT_H, INPUT_W = 512, 768
IN_C, OUT_C      = 3, 2


In [29]:
# --- TiledInferenceModelONNX smoke test ---
INPUT_H_ONNX, INPUT_W_ONNX = 480, 480
IN_C_ONNX, OUT_C_ONNX = 3, 1

class MockBinarySegModel(nn.Module):
    """Simple model that outputs 1-channel sigmoid-like predictions."""
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(IN_C_ONNX, OUT_C_ONNX, kernel_size=1, bias=False)
    def forward(self, x):
        return torch.sigmoid(self.conv(x))


In [30]:

tiled_onnx = TiledInferenceModelONNX(
    model=MockBinarySegModel(),
    tile_size=256, overlap=32,
    input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
    postprocessing="none",
)
print(f"[OK] TiledInferenceModelONNX: tiler tiles={tiled_onnx.tiler.num_tiles}, "
      f"merger tiles={tiled_onnx.merger.num_tiles}")


[OK] TiledInferenceModelONNX: tiler tiles=4, merger tiles=4


In [31]:

x_onnx = torch.rand(1, IN_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX)
with torch.no_grad():
    out_onnx = tiled_onnx(x_onnx)

assert out_onnx.shape == (1, OUT_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX), \
    f"Wrong shape: {out_onnx.shape}"
print(f"[OK] Forward: {tuple(x_onnx.shape)} → {tuple(out_onnx.shape)}")


[OK] Forward: (1, 3, 480, 480) → (1, 1, 480, 480)


In [32]:
# --- TiledInferenceModelONNX: input SMALLER than tile_size ---
SMALL_H, SMALL_W = 100, 80

class SmallMockSeg(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 1, kernel_size=1, bias=False)
    def forward(self, x):
        return torch.sigmoid(self.conv(x))

tiled_small = TiledInferenceModelONNX(
    model=SmallMockSeg(),
    tile_size=256, overlap=32,
    input_h=SMALL_H, input_w=SMALL_W,
    postprocessing="none",
)
assert tiled_small.tiler.num_tiles >= 1
print(f"[OK] TiledInferenceModelONNX (small): tiles={tiled_small.tiler.num_tiles}")

x_small = torch.rand(1, 1, SMALL_H, SMALL_W)
with torch.no_grad():
    out_small = tiled_small(x_small)
assert out_small.shape == (1, 1, SMALL_H, SMALL_W), f"Wrong shape: {out_small.shape}"
print(f"[OK] Forward (small): {tuple(x_small.shape)} → {tuple(out_small.shape)}")


[OK] TiledInferenceModelONNX (small): tiles=1
[OK] Forward (small): (1, 1, 100, 80) → (1, 1, 100, 80)


In [ ]:
# --- Postprocessing presets ---
for preset in ("none", "light", "medium", "heavy"):
    tiled_pp = TiledInferenceModelONNX(
        model=MockBinarySegModel(),
        tile_size=256, overlap=32,
        input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
        postprocessing=preset,
    )
    expected_len = 0 if preset == "none" else 2
    assert len(tiled_pp.merger.postprocessing_pipeline) == expected_len, preset
    with torch.no_grad():
        out_pp = tiled_pp(x_onnx)
    assert out_pp.shape == (1, OUT_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX)
    print(f"[OK] postprocessing={preset!r}  pipeline_steps={len(tiled_pp.merger.postprocessing_pipeline)}")

### DualOutputTiledInferenceModelONNX smoke tests

In [ ]:
# --- Mock dual-output model: seg + presence ---
C_SEG, C_CLS = 1, 3  # 1-ch segmentation, 3-class presence

class MockDualOutputModel(nn.Module):
    """Returns (seg, presence) — simulates a seg+classification head."""
    def __init__(self):
        super().__init__()
        self.seg_head = nn.Conv2d(IN_C_ONNX, C_SEG, kernel_size=1, bias=False)
        self.cls_pool = nn.AdaptiveAvgPool2d(1)
        self.cls_head = nn.Linear(IN_C_ONNX, C_CLS)

    def forward(self, x):
        seg = torch.sigmoid(self.seg_head(x))          # (N, C_SEG, H, W)
        pooled = self.cls_pool(x).flatten(1)            # (N, IN_C)
        presence = self.cls_head(pooled)                # (N, C_CLS)
        return seg, presence

In [ ]:
# --- DualOutput: construction + forward (max aggregation) ---
tiled_dual = DualOutputTiledInferenceModelONNX(
    model=MockDualOutputModel(),
    tile_size=256, overlap=32,
    input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
    presence_aggregation="max",
    postprocessing="none",
)
print(f"[OK] DualOutputTiledInferenceModelONNX: tiles={tiled_dual.tiler.num_tiles}, "
      f"presence_agg={tiled_dual.presence_aggregation!r}")

x_dual = torch.rand(1, IN_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX)
with torch.no_grad():
    seg_out, pres_out = tiled_dual(x_dual)

assert seg_out.shape == (1, C_SEG, INPUT_H_ONNX, INPUT_W_ONNX), \
    f"seg wrong shape: {seg_out.shape}"
assert pres_out.shape == (1, C_CLS), \
    f"presence wrong shape: {pres_out.shape}"
print(f"[OK] Forward (max): seg={tuple(seg_out.shape)}, presence={tuple(pres_out.shape)}")

In [ ]:
# --- DualOutput: mean aggregation ---
tiled_dual_mean = DualOutputTiledInferenceModelONNX(
    model=MockDualOutputModel(),
    tile_size=256, overlap=32,
    input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
    presence_aggregation="mean",
    postprocessing="none",
)

with torch.no_grad():
    seg_mean, pres_mean = tiled_dual_mean(x_dual)

assert seg_mean.shape == (1, C_SEG, INPUT_H_ONNX, INPUT_W_ONNX)
assert pres_mean.shape == (1, C_CLS)
print(f"[OK] Forward (mean): seg={tuple(seg_mean.shape)}, presence={tuple(pres_mean.shape)}")

# --- Verify max >= mean (for positive logits, max should dominate) ---
# Not strictly always true for arbitrary logits, but checks plumbing works
print(f"     max presence:  {pres_out[0].tolist()}")
print(f"     mean presence: {pres_mean[0].tolist()}")

In [ ]:
# --- DualOutput: invalid presence_aggregation raises ---
try:
    DualOutputTiledInferenceModelONNX(
        model=MockDualOutputModel(),
        tile_size=256, overlap=32,
        input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
        presence_aggregation="invalid",
    )
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"[OK] Invalid aggregation raises: {e}")

## Device tests — CPU and GPU

`TiledInferenceModelONNX` must work identically on CPU and CUDA.

- `TileProcessorONNX.forward()` uses only slicing/cat, so output is always on `x.device`.
- `TileMergerONNX` stores its weight maps as `register_buffer` tensors, which move
  automatically when `.to(device)` is called.
- All dynamic tensors inside the merger's forward methods are created with `device=tiles.device`.

Calling `tiled_model.to(device)` propagates to `self.tiler`, `self.model`, and `self.merger`
because all three are proper `nn.Module` attributes.

In [ ]:
def _make_tiled_onnx(device):
    """Build a TiledInferenceModelONNX and move it + a dummy input to `device`."""
    model = MockBinarySegModel().to(device)
    tiled = TiledInferenceModelONNX(
        model=model,
        tile_size=256, overlap=32,
        input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
        postprocessing="none",
    ).to(device)
    x = torch.rand(1, IN_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX, device=device)
    return tiled, x


def _check_device(label, tiled, x):
    """Run forward and assert output device matches input device."""
    with torch.no_grad():
        out = tiled(x)
    assert out.shape == (1, OUT_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX), \
        f"[{label}] Wrong shape: {out.shape}"
    assert out.device.type == x.device.type, \
        f"[{label}] Device mismatch: output={out.device}, input={x.device}"
    # Verify merger buffers are on the same device as the module
    assert tiled.merger.distance_weight.device.type == x.device.type, \
        f"[{label}] merger.distance_weight on wrong device: {tiled.merger.distance_weight.device}"
    assert tiled.merger.gaussian_weight.device.type == x.device.type, \
        f"[{label}] merger.gaussian_weight on wrong device: {tiled.merger.gaussian_weight.device}"
    print(f"[OK] {label}: {tuple(x.shape)} → {tuple(out.shape)}  device={out.device}")

In [ ]:
# --- CPU test ---
tiled_cpu, x_cpu = _make_tiled_onnx("cpu")
_check_device("CPU", tiled_cpu, x_cpu)

# --- GPU test (skipped gracefully when CUDA unavailable) ---
if torch.cuda.is_available():
    tiled_gpu, x_gpu = _make_tiled_onnx("cuda")
    _check_device("GPU", tiled_gpu, x_gpu)
    torch.cuda.synchronize()
    print("[OK] GPU: cuda.synchronize() passed")
else:
    print("[SKIP] GPU test skipped — CUDA not available")

In [ ]:
# --- All merge strategies on CPU ---
STRATEGIES = ["overwrite", "average", "majority_vote", "confidence_weighted", "gaussian_weighted"]
for strategy in STRATEGIES:
    m = MockBinarySegModel()
    tiled = TiledInferenceModelONNX(
        model=m, tile_size=256, overlap=32,
        input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
        merge_strategy=strategy, postprocessing="none",
    )
    with torch.no_grad():
        out = tiled(torch.rand(1, IN_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX))
    assert out.shape == (1, OUT_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX)
    print(f"[OK] strategy={strategy!r}  shape={tuple(out.shape)}")

# --- All strategies on GPU (if available) ---
if torch.cuda.is_available():
    for strategy in STRATEGIES:
        m = MockBinarySegModel().cuda()
        tiled = TiledInferenceModelONNX(
            model=m, tile_size=256, overlap=32,
            input_h=INPUT_H_ONNX, input_w=INPUT_W_ONNX,
            merge_strategy=strategy, postprocessing="none",
        ).cuda()
        with torch.no_grad():
            out = tiled(torch.rand(1, IN_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX, device="cuda"))
        assert out.device.type == "cuda"
        assert out.shape == (1, OUT_C_ONNX, INPUT_H_ONNX, INPUT_W_ONNX)
        print(f"[OK] GPU strategy={strategy!r}  shape={tuple(out.shape)}")
else:
    print("[SKIP] GPU strategy tests skipped — CUDA not available")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()